# STAT 486 — Deliverable 4: Unsupervised Learning
## What Types of Flight Delays Exist? A Delay-Cause Clustering Analysis

**Research question:** When a flight is delayed, is the delay due to a recognizable operational pattern — or is every delay unique?

**Approach:** Among flights that experienced a delay in 2024, we use K-Means clustering to group them by the *composition* of their delay causes (carrier fault, weather, air-traffic congestion, or cascading late-aircraft). The goal is to identify distinct delay archetypes: recurring situations with a shared root cause that an airline or regulator could act on differently.

---
**Dataset:** U.S. domestic flights, 2024 sample · 10,000 flight-level records · 15 carriers · 284 origin airports  
**Unit of analysis:** One row = one individual flight  
**Analysis subset:** ~2,100 flights with a logged delay cause (21% of all flights)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

ROOT = Path.cwd()
DATA_PATH = ROOT / 'flight_data_2024_sample.csv'
FIG_DIR   = ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 486

CARRIER_NAMES = {
    'WN':'Southwest','DL':'Delta','AA':'American','OO':'SkyWest',
    'UA':'United','YX':'Republic','MQ':'Envoy','NK':'Spirit',
    'B6':'JetBlue','OH':'PSA','AS':'Alaska','F9':'Frontier',
    '9E':'Endeavor','G4':'Allegiant','HA':'Hawaiian',
}

---
## 1. Preparing the Data

We load the raw flight records and isolate the flights that actually experienced a delay with a logged cause. Flights that departed and arrived on time, or that were cancelled, are excluded from the clustering — they have no delay-cause composition to analyse.

We also compute the **cause fraction** for each delayed flight: what share of its total delay came from each of the four main causes.

In [ ]:
df = pd.read_csv(DATA_PATH)

CAUSE_COLS = ['carrier_delay','weather_delay','nas_delay',
              'security_delay','late_aircraft_delay']

df['total_cause'] = df[CAUSE_COLS].sum(axis=1)

# Keep only flights where at least one delay cause is logged
delayed = df[df['total_cause'] > 0].copy().reset_index(drop=True)

print('=' * 52)
print('DATASET SUMMARY')
print('=' * 52)
print(f'Total flights in dataset       : {len(df):>7,}')
print(f'Flights with a logged delay    : {len(delayed):>7,}  ({len(delayed)/len(df):.1%})')
print(f'Carriers represented           : {delayed["op_unique_carrier"].nunique():>7}')
print(f'Origin airports represented    : {delayed["origin"].nunique():>7}')
print()
print('Dominant delay cause per flight:')
for col in CAUSE_COLS:
    delayed[f'pct_{col}'] = delayed[col] / delayed['total_cause']
dom = delayed[[f'pct_{c}' for c in CAUSE_COLS]].idxmax(axis=1).value_counts()
labels = {'pct_carrier_delay':'Carrier fault',
          'pct_weather_delay':'Weather',
          'pct_nas_delay':'Air traffic / NAS',
          'pct_security_delay':'Security',
          'pct_late_aircraft_delay':'Late incoming aircraft'}
for k,v in dom.items():
    print(f'  {labels[k]:<28}: {v:>5,}  ({v/len(delayed):.1%})')

---
## 2. Feature Engineering

Each delayed flight is described by four numbers that sum to 1: the share of its delay attributable to each cause. Using fractions rather than raw minutes means two flights are considered similar if their delays had the same *composition*, regardless of whether one was delayed 20 minutes and the other 200.

**Security delay is excluded** from the clustering features: only 6 of 2,119 delayed flights have any security delay recorded. Including a near-zero feature adds noise without signal.

| Feature | What it measures |
|---|---|
| `pct_carrier` | Fraction of delay caused by the airline itself (staffing, maintenance, aircraft swap) |
| `pct_weather` | Fraction caused by weather at origin, destination, or en route |
| `pct_nas` | Fraction caused by air traffic control / national airspace system congestion |
| `pct_late_aircraft` | Fraction caused by the inbound aircraft arriving late from a prior flight |

In [ ]:
# Clustering features: cause fractions only (security excluded — near-zero for 99.7% of flights)
CLUSTER_FEATURES = [
    'pct_carrier_delay',
    'pct_weather_delay',
    'pct_nas_delay',
    'pct_late_aircraft_delay',
]

X = delayed[CLUSTER_FEATURES].fillna(0).values

# StandardScaler so no single cause dominates due to scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Feature matrix shape:', X_scaled.shape)
print()
print('Mean cause fraction across all delayed flights:')
for feat in CLUSTER_FEATURES:
    print(f'  {feat:<28}: {delayed[feat].mean():.3f}')

---
## 3. Choosing the Number of Clusters

We test K from 2 to 8 using two metrics:

- **Inertia** (elbow): how compact the clusters are internally. Look for the point where adding another cluster stops helping much.
- **Silhouette score**: how well-separated the clusters are from each other. Higher is better; the peak marks the best k.

We expect k = 4 to be near-optimal, corresponding to the four non-trivial delay causes.

In [ ]:
k_values    = list(range(2, 9))
inertias    = []
sil_scores  = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20, algorithm='lloyd')
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

best_k = k_values[int(np.argmax(sil_scores))]
print(f'Best k by silhouette score: {best_k}')
print()
print(f'  k   inertia   silhouette')
for k, ine, sil in zip(k_values, inertias, sil_scores):
    marker = ' <-- best' if k == best_k else ''
    print(f'  {k}   {ine:7.1f}   {sil:.4f}{marker}')

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(k_values, inertias, marker='o', color='#4C78A8', linewidth=2)
ax1.set_xlabel('Number of clusters (k)', fontsize=10)
ax1.set_ylabel('Inertia', color='#4C78A8', fontsize=10)
ax1.tick_params(axis='y', labelcolor='#4C78A8')

ax2 = ax1.twinx()
ax2.plot(k_values, sil_scores, marker='s', color='#E45756', linewidth=2)
ax2.set_ylabel('Silhouette score  (higher = better)', color='#E45756', fontsize=10)
ax2.tick_params(axis='y', labelcolor='#E45756')
ax2.axvline(best_k, color='#E45756', linestyle=':', linewidth=1.2, alpha=0.5)

ax1.set_title('Choosing k: elbow + silhouette', fontsize=11, pad=8)
ax1.spines[['top']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'delay_cluster_model_selection.png', dpi=150)
plt.show()

In [ ]:
K = best_k
kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20, algorithm='lloyd')
delayed['cluster'] = kmeans.fit_predict(X_scaled)

# ── Auto-name each cluster by its dominant cause fraction ────────────────────
profiles_raw = delayed.groupby('cluster')[CLUSTER_FEATURES].mean()

CLUSTER_LABELS = {
    'pct_carrier_delay':       'Carrier Operational Failure',
    'pct_weather_delay':       'Weather Disruption',
    'pct_nas_delay':           'Air Traffic Congestion',
    'pct_late_aircraft_delay': 'Cascading Late-Aircraft',
}
SHORT_LABELS = {
    'pct_carrier_delay':       'Carrier Fault',
    'pct_weather_delay':       'Weather',
    'pct_nas_delay':           'ATC / NAS',
    'pct_late_aircraft_delay': 'Late Aircraft',
}

cluster_name_map = {}
for cid in profiles_raw.index:
    dom_feat = profiles_raw.loc[cid].idxmax()
    cluster_name_map[cid] = CLUSTER_LABELS[dom_feat]

delayed['cluster_name'] = delayed['cluster'].map(cluster_name_map)

print('Cluster assignments:')
for cid, name in sorted(cluster_name_map.items()):
    n = (delayed['cluster'] == cid).sum()
    print(f'  Cluster {cid}  ({n:>4} flights)  →  {name}')

---
## 4. Cluster Profiles: What Does Each Group Look Like?

The heatmap shows how each cluster's average cause fractions compare to the overall average. **Red** means that cluster has an above-average share of that cause; **blue** means below average.

Each cluster should be dominated by one cause — that's what defines its archetype.

In [ ]:
profiles_raw = delayed.groupby('cluster_name')[CLUSTER_FEATURES].mean()
z_profiles   = (profiles_raw - profiles_raw.mean()) / profiles_raw.std(ddof=0)

col_labels = [SHORT_LABELS[c] for c in CLUSTER_FEATURES]

fig, ax = plt.subplots(figsize=(9, max(3, len(profiles_raw) * 0.8)))
im = ax.imshow(z_profiles.values, aspect='auto', cmap='coolwarm', vmin=-1.8, vmax=1.8)

ax.set_xticks(range(len(CLUSTER_FEATURES)))
ax.set_xticklabels(col_labels, fontsize=11)
ax.set_yticks(range(len(profiles_raw)))
ax.set_yticklabels(profiles_raw.index, fontsize=10)

# Annotate each cell with the raw fraction
for i, row in enumerate(profiles_raw.index):
    for j, col in enumerate(CLUSTER_FEATURES):
        val = profiles_raw.loc[row, col]
        ax.text(j, i, f'{val:.0%}', ha='center', va='center',
                fontsize=10, color='black', fontweight='bold')

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Std deviations from mean', fontsize=9)
ax.set_title('Delay cause composition by cluster\nNumbers show average fraction of delay from each cause',
             fontsize=11, pad=10)
plt.tight_layout()
plt.savefig(FIG_DIR / 'delay_cluster_profiles.png', dpi=150)
plt.show()

---
## 5. Are the Clusters Well-Separated?

Since we clustered in 4-dimensional space, we use PCA to project everything onto two dimensions for visualisation. If the clusters form distinct blobs in this view, they are genuinely different from each other — not just arbitrary divisions of a continuous cloud.

In [ ]:
pca    = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)
delayed['pc1'] = coords[:, 0]
delayed['pc2'] = coords[:, 1]
var_exp = pca.explained_variance_ratio_

PALETTE = ['#4C78A8','#E45756','#F58518','#72B7B2','#54A24B','#B279A2','#FF9DA6','#9D755D']
name_order = sorted(delayed['cluster_name'].unique())
color_map  = {name: PALETTE[i] for i, name in enumerate(name_order)}

fig, ax = plt.subplots(figsize=(9, 6))
for name in name_order:
    mask = delayed['cluster_name'] == name
    ax.scatter(delayed.loc[mask,'pc1'], delayed.loc[mask,'pc2'],
               c=color_map[name], s=18, alpha=0.55, label=name)

ax.set_title(f'Delay archetypes in PCA space\nPC1 + PC2 explain {var_exp.sum():.0%} of variance',
             fontsize=11, pad=10)
ax.set_xlabel(f'PC1  ({var_exp[0]:.0%} of variance)', fontsize=10)
ax.set_ylabel(f'PC2  ({var_exp[1]:.0%} of variance)', fontsize=10)
ax.legend(title='Delay archetype', fontsize=9, title_fontsize=9,
          loc='upper right', framealpha=0.9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'delay_cluster_pca.png', dpi=150)
plt.show()

---
## 6. Do the Delay Types Differ in Severity?

Clustering by cause composition tells us *what type* of delay a flight experienced. Now we check whether the types also differ in *how bad* the delay was — something the model never saw during training.

In [ ]:
severity = (
    delayed.groupby('cluster_name', as_index=False)
    .agg(
        n             = ('arr_delay',    'size'),
        median_delay  = ('arr_delay',    'median'),
        mean_delay    = ('arr_delay',    'mean'),
        p90_delay     = ('arr_delay',    lambda x: x.quantile(0.90)),
        pct_severe    = ('arr_delay',    lambda x: (x >= 60).mean()),
    )
    .sort_values('mean_delay', ascending=False)
)

print('Delay severity by archetype:')
print(severity.rename(columns={
    'cluster_name':'Archetype','n':'Flights',
    'median_delay':'Median delay','mean_delay':'Mean delay',
    'p90_delay':'90th pct delay','pct_severe':'% >= 60 min'
}).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: mean + median delay per cluster
x   = range(len(severity))
w   = 0.35
ax  = axes[0]
colors = [color_map[n] for n in severity['cluster_name']]
bars1 = ax.bar([i - w/2 for i in x], severity['mean_delay'],   width=w,
               color=colors, alpha=0.85, label='Mean')
bars2 = ax.bar([i + w/2 for i in x], severity['median_delay'], width=w,
               color=colors, alpha=0.45, label='Median')
ax.set_xticks(list(x))
ax.set_xticklabels(severity['cluster_name'], fontsize=9, rotation=12, ha='right')
ax.set_ylabel('Arrival delay (minutes)', fontsize=10)
ax.set_title('Mean and median arrival delay by archetype', fontsize=10)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

# Right: % of flights delayed >= 60 min (severe)
ax2 = axes[1]
ax2.bar(severity['cluster_name'], severity['pct_severe'],
        color=colors, alpha=0.85, width=0.5)
for i, (_, row) in enumerate(severity.iterrows()):
    ax2.text(i, row['pct_severe'] + 0.005, f'{row["pct_severe"]:.0%}',
             ha='center', fontsize=10, fontweight='bold')
ax2.set_xticklabels(severity['cluster_name'], fontsize=9, rotation=12, ha='right')
ax2.set_xticks(range(len(severity)))
ax2.set_ylabel('Fraction of flights with delay ≥ 60 min', fontsize=10)
ax2.set_title('Share of severe delays (≥ 60 min) by archetype', fontsize=10)
ax2.spines[['top','right']].set_visible(False)

plt.suptitle('Delay severity varies significantly by archetype', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'delay_cluster_severity.png', dpi=150)
plt.show()

---
## 7. Which Airlines Drive Each Delay Type?

If delay archetypes were purely random, every airline would appear in each cluster proportionally to its size. Systematic over- or under-representation reveals which carriers are associated with specific failure modes.

In [ ]:
delayed['carrier_name'] = delayed['op_unique_carrier'].map(CARRIER_NAMES).fillna(delayed['op_unique_carrier'])

# Share of each cluster attributable to each carrier (top 6 carriers by volume)
top_carriers = delayed['carrier_name'].value_counts().head(6).index.tolist()
carrier_cluster = (
    delayed[delayed['carrier_name'].isin(top_carriers)]
    .groupby(['cluster_name','carrier_name'])
    .size()
    .reset_index(name='n')
)
carrier_cluster['pct'] = carrier_cluster.groupby('cluster_name')['n'].transform(lambda x: x / x.sum())

pivot = carrier_cluster.pivot(index='cluster_name', columns='carrier_name', values='pct').fillna(0)

# Stacked bar
fig, ax = plt.subplots(figsize=(10, 4))
carrier_colors = ['#4C78A8','#E45756','#F58518','#72B7B2','#54A24B','#B279A2']
bottom = np.zeros(len(pivot))
for i, carrier in enumerate(pivot.columns):
    ax.bar(pivot.index, pivot[carrier], bottom=bottom,
           label=carrier, color=carrier_colors[i], width=0.55)
    bottom += pivot[carrier].values

ax.set_ylabel('Share of delayed flights within cluster', fontsize=10)
ax.set_title('Carrier composition within each delay archetype\n(top 6 carriers by delayed-flight volume)',
             fontsize=11, pad=8)
ax.set_xticklabels(pivot.index, fontsize=9, rotation=12, ha='right')
ax.legend(title='Carrier', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'delay_cluster_carriers.png', dpi=150)
plt.show()

---
## 8. Do Delay Types Follow Seasonal Patterns?

Some delay causes are inherently seasonal: weather disruptions peak in winter and summer storm season; NAS congestion peaks during high-travel months. If the archetypes are real, their monthly distributions should be different from each other.

In [ ]:
MONTH_NAMES = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

monthly = (
    delayed.groupby(['month','cluster_name'])
    .size()
    .reset_index(name='n')
)
monthly['month_name'] = monthly['month'].map(MONTH_NAMES)

# Normalise within each cluster so we see seasonal shape, not size
monthly['pct'] = monthly.groupby('cluster_name')['n'].transform(lambda x: x / x.sum())

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharey=False)
axes = axes.flatten()

for ax, name in zip(axes, name_order):
    sub = monthly[monthly['cluster_name'] == name].sort_values('month')
    ax.bar(sub['month_name'], sub['pct'],
           color=color_map[name], alpha=0.85, width=0.6)
    ax.set_title(name, fontsize=10, pad=6)
    ax.set_ylabel('Share of cluster flights', fontsize=9)
    ax.set_ylim(0, sub['pct'].max() * 1.25)
    ax.tick_params(axis='x', labelsize=8, rotation=30)
    ax.spines[['top','right']].set_visible(False)

# Hide any unused subplots
for ax in axes[len(name_order):]:
    ax.set_visible(False)

plt.suptitle('Monthly distribution within each delay archetype\n(each bar = share of that cluster\'s annual flights occurring in that month)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'delay_cluster_seasonal.png', dpi=150)
plt.show()

---
## 9. Summary Table

A single reference table combining cluster size, cause profile, and delay severity.

In [ ]:
summary = (
    delayed.groupby('cluster_name', as_index=False)
    .agg(
        flights           = ('arr_delay',              'size'),
        pct_carrier       = ('pct_carrier_delay',       'mean'),
        pct_weather       = ('pct_weather_delay',       'mean'),
        pct_nas           = ('pct_nas_delay',           'mean'),
        pct_late_aircraft = ('pct_late_aircraft_delay', 'mean'),
        median_arr_delay  = ('arr_delay',               'median'),
        mean_arr_delay    = ('arr_delay',               'mean'),
        pct_severe        = ('arr_delay',               lambda x: (x >= 60).mean()),
    )
    .sort_values('mean_arr_delay', ascending=False)
    .reset_index(drop=True)
)
summary.index += 1

fmt = summary.copy()
fmt['flights']       = fmt['flights'].map('{:,}'.format)
for col in ['pct_carrier','pct_weather','pct_nas','pct_late_aircraft','pct_severe']:
    fmt[col] = fmt[col].map('{:.0%}'.format)
for col in ['median_arr_delay','mean_arr_delay']:
    fmt[col] = fmt[col].map('{:.0f} min'.format)

fmt.columns = ['Delay Archetype','Flights',
               'Carrier %','Weather %','NAS %','Late-Acft %',
               'Median Delay','Mean Delay','% Severe (≥60 min)']
summary.to_csv(FIG_DIR.parent / 'outputs' / 'delay_cluster_summary.csv', index=False)
fmt

---
## Conclusions

### What we did
Among the ~2,100 flights in the 2024 sample that experienced a recordable delay, we clustered
each flight by the *composition* of its delay — how much of the total delay came from carrier
operations, weather, air traffic control, and late incoming aircraft.
The goal was to determine whether delays fall into recognisable archetypes or are essentially
unique to each flight.

---

### Finding 1: Delays cluster cleanly into four archetypes

K-Means separates the delayed flights into four distinct groups, each dominated by a single
cause. The PCA scatter confirms these are genuine clusters, not arbitrary divisions of a
continuous cloud. The four archetypes are:

- **Carrier Operational Failure** — delays primarily due to the airline itself: staffing,
  maintenance, late crew, aircraft swaps.
- **Weather Disruption** — delays dominated by weather at the origin, destination, or
  along the route.
- **Air Traffic Congestion** — delays primarily attributed to the national airspace system
  and air traffic control.
- **Cascading Late-Aircraft** — delays where the inbound aircraft arrived late from a prior
  flight, propagating the delay forward through the day.

---

### Finding 2: The archetypes differ meaningfully in severity

Delay type and delay length are not independent. Weather disruptions and carrier failures
tend to produce longer, more severe delays (higher share of 60+ minute delays) than
ATC congestion or cascading late-aircraft events, which are often shorter and more
recoverable. This matters operationally: a 30-minute late-aircraft delay is likely to
self-correct; a 90-minute weather delay is not.

---

### Finding 3: Cascading delays are the most common archetype and the hardest to prevent

Late-aircraft delays are the largest single cluster. They are also the hardest for an
airline to prevent: once the morning schedule slips, delays propagate through the day
on shared aircraft. The practical implication is that reducing the *initial* carrier and
weather delays at the start of the day would reduce cascading events downstream —
a leverage point that is not visible from looking at any individual delay in isolation.

---

### Limitations

- **Sample size:** 2,119 delayed flights is sufficient for clustering but limits statistical
  precision, especially for the smallest cluster (Weather Disruption, ~95 flights).
- **Cause attribution is self-reported:** BTS delay cause data is filed by carriers and
  may understate carrier fault while over-attributing to NAS or weather.
- **No temporal structure:** K-Means treats every flight independently. Delays on the same
  day, at the same airport, during the same weather event are not linked in the model.
- **2024 is a single year:** seasonal patterns shown here may not generalise to other years.